# Semana 10: Espacios Vectoriales y Transformaciones Lineales
## Notebook de Ejercicios (NB2) - Visualización de Fashion-MNIST con PCA

### Propósito de la sesión
Aplicar técnicas de reducción de dimensionalidad (PCA) para visualizar el espacio de características de Fashion-MNIST en 2D. A partir de esta visualización, discutiremos por qué se necesitan **redes profundas** (múltiples transformaciones lineales + no-linealidades) para clasificar correctamente este tipo de datos.

### Objetivos de aprendizaje
*   Cargar y explorar el dataset **Fashion-MNIST**.
*   Aplicar **PCA (Análisis de Componentes Principales)** para reducir la dimensionalidad de 784 a 2 dimensiones.
*   Visualizar las imágenes proyectadas en 2D, coloreadas por clase.
*   Analizar la separabilidad de las clases en el espacio 2D.
*   Discutir por qué una **red profunda** (con capas ocultas y no-linealidades) es necesaria para lograr una buena clasificación.

## Configuración Inicial

Importamos las librerías necesarias: torchvision para Fashion-MNIST, sklearn para PCA, y matplotlib para visualizaciones.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.datasets import fetch_openml

import torch
from torchvision import datasets, transforms

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 10)

# Fijamos semilla para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. Carga del Dataset Fashion-MNIST

Fashion-MNIST es un dataset de imágenes de ropa y accesorios, con 10 clases. Cada imagen es de 28x28 píxeles (784 dimensiones).

In [ ]:
# Cargamos Fashion-MNIST usando torchvision
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# Nombres de las clases
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"🔷 Fashion-MNIST cargado correctamente.")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Test samples: {len(test_dataset)}")
print(f"  Dimensiones de cada imagen: {train_dataset[0][0].shape}")

In [ ]:
# Visualizamos algunas imágenes
def show_fashion_mnist_samples(dataset, class_names, num_samples=10):
    plt.figure(figsize=(15, 6))
    for i in range(num_samples):
        plt.subplot(2, 5, i+1)
        img, label = dataset[i]
        plt.imshow(img.squeeze(), cmap='gray')
        plt.title(f'{class_names[label]}')
        plt.axis('off')
    plt.suptitle('Muestras de Fashion-MNIST', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

show_fashion_mnist_samples(train_dataset, class_names)

---
## 2. Preparación de los Datos para PCA

Para aplicar PCA, necesitamos una matriz $X$ de forma `(n_muestras, n_características)`. Aplanaremos las imágenes de 28x28 a vectores de 784 dimensiones.

In [ ]:
# Tomamos un subconjunto del dataset para que PCA sea más rápido (5000 imágenes)
n_samples = 5000
indices = np.random.choice(len(train_dataset), n_samples, replace=False)

X = []
y = []
for idx in indices:
    img, label = train_dataset[idx]
    X.append(img.numpy().flatten())  # aplanamos
    y.append(label)

X = np.array(X)
y = np.array(y)

print(f"Shape de X: {X.shape} ({n_samples} muestras, 784 características)")
print(f"Shape de y: {y.shape}")
print(f"Clases presentes: {np.unique(y)}")

---
## Actividad 1: Visualizar las imágenes en 2D usando PCA

### 2.1. Aplicación de PCA

PCA encuentra las direcciones de máxima varianza y proyecta los datos en un espacio de menor dimensión.

In [ ]:
# Aplicamos PCA para reducir a 2 dimensiones
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

print(f"Shape después de PCA: {X_pca.shape}")
print(f"Varianza explicada por componente: {pca.explained_variance_ratio_}")
print(f"Varianza total explicada: {sum(pca.explained_variance_ratio_):.4f}")

# Visualizamos la varianza explicada
plt.figure(figsize=(8, 5))
plt.bar(['PC1', 'PC2'], pca.explained_variance_ratio_)
plt.ylabel('Proporción de varianza explicada')
plt.title('Varianza explicada por cada componente principal')
plt.show()

### 2.2. Visualización de la proyección 2D

In [ ]:
plt.figure(figsize=(14, 10))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter, ticks=range(10), label='Clase')
plt.xlabel('Primera Componente Principal')
plt.ylabel('Segunda Componente Principal')
plt.title('Proyección PCA 2D de Fashion-MNIST')

# Añadimos etiquetas de clase en el centroide de cada cluster
for i in range(10):
    centroid = X_pca[y == i].mean(axis=0)
    plt.annotate(class_names[i], centroid, fontsize=12, weight='bold',
                 ha='center', va='center', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.show()

### 2.3. Análisis de la separabilidad

Observamos que:
*   Algunas clases están relativamente bien separadas (ej. 'Trouser' - pantalones, 'Sneaker' - zapatillas).
*   Otras clases se superponen considerablemente (ej. 'Shirt' - camisa, 'Coat' - abrigo, 'Pullover' - suéter).
*   Con solo 2 dimensiones (PCA), **no es posible separar linealmente** todas las clases.

### 2.4. Visualización con miniaturas de imágenes (opcional)

Para una visualización más intuitiva, podemos mostrar las imágenes reales en lugar de puntos.

In [ ]:
def plot_images_on_pca(X_pca, X_original, y, class_names, num_images=100):
    fig, ax = plt.subplots(figsize=(16, 12))
    ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.3, s=5, c='gray')
    
    # Seleccionamos algunas imágenes aleatorias para mostrar
    indices = np.random.choice(len(X_pca), num_images, replace=False)
    
    for idx in indices:
        x, y_coord = X_pca[idx]
        img = X_original[idx].reshape(28, 28)
        imagebox = OffsetImage(img, zoom=0.5, cmap='gray')
        ab = AnnotationBbox(imagebox, (x, y_coord), frameon=False)
        ax.add_artist(ab)
    
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title('Visualización de Fashion-MNIST con miniaturas')
    plt.show()

# Llamar a la función (puede ser lenta, usamos solo 100 imágenes)
plot_images_on_pca(X_pca, X, y, class_names, num_images=100)

---
## Actividad 2: Discutir por qué se necesita una red profunda

### 2.1. Limitaciones de los modelos lineales

Un modelo lineal (como regresión logística) equivale a trazar **hiperplanos de separación** en el espacio original. En el espacio PCA 2D, un clasificador lineal sería una línea recta. Dada la superposición de clases, una línea no puede separar adecuadamente todas las categorías.

### 2.2. ¿Por qué PCA no es suficiente?

PCA es una **transformación lineal** (ortogonal). Como vimos en el NB1, las transformaciones lineales no pueden desenredar estructuras no lineales. Las clases que se superponen en el espacio PCA 2D reflejan que la estructura subyacente de los datos no es linealmente separable.

### 2.3. Necesidad de redes profundas

Las redes profundas apilan **múltiples transformaciones lineales** intercaladas con **no-linealidades** (ReLU, sigmoide, etc.). Esto permite:

1.  **Transformar el espacio** de manera no lineal, "desenredando" las clases.
2.  **Aprender características jerárquicas**: de píxeles a bordes, a formas, a partes de objetos.
3.  **Lograr separabilidad** en un espacio de características aprendido (no fijo como PCA).

En la práctica, una red con suficientes capas ocultas puede aprender una representación donde las clases sean (casi) linealmente separables en la última capa.

In [ ]:
# Simulación conceptual: una red profunda aprende una transformación no lineal
# No podemos visualizar directamente el espacio interno, pero podemos mostrar
# cómo una transformación no lineal podría "separar" las clases.

# Tomamos los datos PCA y aplicamos una transformación no lineal simulada
# (esto es solo ilustrativo, no es lo que hace una red real)
def nonlinear_transform(x):
    # Aplicamos una transformación no lineal arbitraria
    r = np.sqrt(x[:, 0]**2 + x[:, 1]**2)
    theta = np.arctan2(x[:, 1], x[:, 0])
    # Transformación: separar por radio y ángulo
    x_new = np.column_stack([
        r * np.cos(2*theta),  # frecuencia doble
        r * np.sin(2*theta)
    ])
    return x_new

X_transformed = nonlinear_transform(X_pca)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', alpha=0.6, s=5)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Espacio PCA original')

plt.subplot(1, 2, 2)
plt.scatter(X_transformed[:, 0], X_transformed[:, 1], c=y, cmap='tab10', alpha=0.6, s=5)
plt.xlabel('dimensión 1')
plt.ylabel('dimensión 2')
plt.title('Después de transformación no lineal (simulada)')

plt.suptitle('Efecto de una transformación no lineal en la separabilidad')
plt.tight_layout()
plt.show()

print("📌 Una transformación no lineal puede desenredar las clases, haciéndolas más separables.")
print("📌 Esto es lo que las redes profundas aprenden automáticamente.")

### 2.4. Ejemplo con una red neuronal simple

Para reforzar la idea, podemos entrenar una red neuronal simple en Fashion-MNIST y comparar con un modelo lineal.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Usamos un subconjunto para rapidez
X_flat = X  # ya aplanado

# Modelo lineal (regresión logística)
print("🔷 Entrenando regresión logística...")
lr = LogisticRegression(max_iter=1000, multi_class='multinomial')
lr.fit(X_flat[:3000], y[:3000])
y_pred_lr = lr.predict(X_flat[3000:])
acc_lr = accuracy_score(y[3000:], y_pred_lr)

# Red neuronal con una capa oculta
print("🔶 Entrenando MLP con una capa oculta...")
mlp = MLPClassifier(hidden_layer_sizes=(100,), activation='relu', max_iter=500)
mlp.fit(X_flat[:3000], y[:3000])
y_pred_mlp = mlp.predict(X_flat[3000:])
acc_mlp = accuracy_score(y[3000:], y_pred_mlp)

print(f"\n📊 Accuracy en test:")
print(f"  Regresión Logística (lineal): {acc_lr:.4f}")
print(f"  MLP (1 capa oculta): {acc_mlp:.4f}")

# Discusión
print("\n📌 La red neuronal supera al modelo lineal porque puede aprender transformaciones no lineales.")
print("📌 Con más capas (profundidad), se pueden aprender representaciones aún más complejas.")

---
## Ejercicios para el Estudiante

1.  **PCA con más componentes:** Aplica PCA para reducir a 3 dimensiones y visualiza (si es posible) en 3D. ¿Mejora la separabilidad?

2.  **t-SNE:** Prueba **t-SNE** en lugar de PCA (`sklearn.manifold.TSNE`). t-SNE es una técnica no lineal de reducción de dimensionalidad. Compara la visualización con PCA.

3.  **Clasificación en espacio PCA:** Entrena un clasificador lineal en el espacio PCA 2D y compara su accuracy con el clasificador en el espacio original. ¿Qué concluyes?

4.  **Profundidad vs. anchura:** Experimenta con MLP de diferentes profundidades (1, 2, 3 capas ocultas) y anchuras (10, 50, 200 neuronas). ¿Cómo afecta a la accuracy?

5.  **Reflexión:** ¿Por qué crees que clases como 'Shirt' son difíciles de separar incluso para redes profundas? ¿Qué características comparten con otras clases?

---
## Conclusiones de la Sesión

*   Hemos aplicado **PCA** para visualizar Fashion-MNIST en 2D, observando que las clases no son linealmente separables en este espacio reducido.
*   PCA, al ser una transformación lineal, no puede desenredar la estructura no lineal de los datos.
*   La superposición de clases en la visualización 2D explica por qué un modelo lineal (como regresión logística) tiene un rendimiento limitado.
*   Las **redes profundas**, al componer múltiples transformaciones lineales con no-linealidades, pueden aprender representaciones donde las clases se vuelven separables.
*   Experimentamos con un MLP simple, confirmando que supera al modelo lineal en accuracy.
*   Este ejercicio conecta los conceptos de **transformaciones lineales** y **espacios vectoriales** con la necesidad de **profundidad** en redes neuronales.

¡Ahora comprendes por qué las redes profundas son tan efectivas para datos complejos como imágenes de moda!